# Clase 144 — Flash Attention + RoPE + GQA

Los 3 ingredientes que hacen que un LLM moderno (Llama-3, Mistral) corra en GPU consumer.
Todo implementado en numpy puro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

n, d = 32, 64
Q = np.random.randn(n, d) / np.sqrt(d)
K = np.random.randn(n, d) / np.sqrt(d)
V = np.random.randn(n, d)
print('Q,K,V shapes:', Q.shape, K.shape, V.shape)

## 1. Atención naive (Q·K^T → softmax → ·V)

In [ ]:
def attention_naive(Q, K, V):
    scores = Q @ K.T / np.sqrt(Q.shape[-1])   # (n, n)
    # softmax row-wise estable
    scores -= scores.max(axis=-1, keepdims=True)
    p = np.exp(scores)
    p /= p.sum(axis=-1, keepdims=True)
    return p @ V, p

out_naive, attn = attention_naive(Q, K, V)
print(f'out shape: {out_naive.shape}, attn entropy mean: {(-(attn*np.log(attn+1e-9)).sum(-1)).mean():.3f}')
print(f'memoria de la matriz attn: {attn.nbytes} bytes')

## 2. Flash Attention (tiling block-by-block)

La idea clave: nunca materializamos la matriz `(n, n)` completa. Procesamos K, V por bloques y mantenemos un running softmax.

In [ ]:
def attention_flash(Q, K, V, block=8):
    n, d = Q.shape
    O = np.zeros_like(V); L = np.zeros((n, 1)); M = np.full((n, 1), -np.inf)
    scale = 1.0 / np.sqrt(d)
    for j in range(0, n, block):
        Kj = K[j:j+block]; Vj = V[j:j+block]
        s = (Q @ Kj.T) * scale                    # (n, block) — bloque pequeño
        mj = s.max(axis=-1, keepdims=True)
        m_new = np.maximum(M, mj)
        # rescale acumulador con shift de softmax
        alpha = np.exp(M - m_new)
        beta = np.exp(s - m_new)
        L = L * alpha + beta.sum(axis=-1, keepdims=True)
        O = O * alpha + beta @ Vj
        M = m_new
    return O / L

out_flash = attention_flash(Q, K, V, block=8)
print(f'flash vs naive max diff: {np.abs(out_flash - out_naive).max():.2e}')
print(f'memoria pico Flash (1 bloque): {Q.shape[0] * 8 * 8} bytes vs naive {attn.nbytes}')

## 3. RoPE (Rotary Position Embeddings)

Rota pares (Q_2i, Q_2i+1) por ángulo `pos * theta_i`. Propiedad: `<RoPE(q,m), RoPE(k,n)> = f(q, k, m-n)` → solo depende de distancia relativa.

In [ ]:
def rope(x, base=10000):
    n, d = x.shape
    assert d % 2 == 0
    half = d // 2
    theta = 1.0 / (base ** (np.arange(half) / half))
    pos = np.arange(n)[:, None]
    angles = pos * theta[None, :]                # (n, d/2)
    cos, sin = np.cos(angles), np.sin(angles)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    out = np.empty_like(x)
    out[..., 0::2] = x1 * cos - x2 * sin
    out[..., 1::2] = x1 * sin + x2 * cos
    return out

Qr = rope(Q); Kr = rope(K)
print('RoPE aplicado; shapes:', Qr.shape, Kr.shape)
out_rope, _ = attention_naive(Qr, Kr, V)
print(f'rope output mean: {out_rope.mean():.4f}')

## 4. Verificar dependencia solo de distancia relativa

In [ ]:
# Crear q y k constantes, ver que <RoPE(q,m), RoPE(k,n)> depende solo de m-n
q = np.ones((1, d)) * 0.1
k = np.ones((1, d)) * 0.1
results = {}
for m, n_pos in [(0, 5), (3, 8), (10, 15), (0, 1), (5, 6)]:
    qr = rope(np.tile(q, (m+1, 1)))[m:m+1]
    kr = rope(np.tile(k, (n_pos+1, 1)))[n_pos:n_pos+1]
    dp = (qr @ kr.T)[0, 0]
    rel = n_pos - m
    results.setdefault(rel, []).append(dp)
for rel, vals in sorted(results.items()):
    print(f'distancia relativa = {rel}: dot products = {[f"{v:.6f}" for v in vals]} (deberían ser iguales)')

## 5. GQA (Grouped Query Attention)

MHA: cada head tiene su propio (K, V). KV cache = `n_layers * 2 * seq * n_heads * d_head`.
GQA: agrupamos `n_heads` queries en `n_kv_heads` grupos que comparten KV. Llama-3 usa `n_heads=32, n_kv_heads=8`.

In [ ]:
n_heads = 8; n_kv_heads = 2; d_head = 16; seq = 32
# MHA full
K_mha = np.random.randn(n_heads, seq, d_head); V_mha = np.random.randn(n_heads, seq, d_head)
mem_mha = K_mha.nbytes + V_mha.nbytes

# GQA: solo n_kv_heads sets, repetidos n_heads/n_kv_heads veces
K_gqa = np.random.randn(n_kv_heads, seq, d_head); V_gqa = np.random.randn(n_kv_heads, seq, d_head)
mem_gqa = K_gqa.nbytes + V_gqa.nbytes

print(f'MHA KV cache: {mem_mha:,} bytes')
print(f'GQA KV cache: {mem_gqa:,} bytes')
print(f'reducción: {mem_mha / mem_gqa:.1f}x  →  {(1 - mem_gqa/mem_mha)*100:.1f}% menos memoria')

# Group expand: cada grupo sirve n_heads/n_kv_heads heads
group_size = n_heads // n_kv_heads
K_expanded = np.repeat(K_gqa, group_size, axis=0)   # (n_heads, seq, d_head)
print(f'K expandido para atención: {K_expanded.shape}')

## Conclusiones

- **Flash Attention**: tiling + online softmax → O(n) memoria en lugar de O(n²). Hace que seq=128k entre en GPU.
- **RoPE**: position embedding multiplicativo → generaliza a secuencias más largas que las vistas en training (con interp/extrapolación).
- **GQA**: trade-off entre MHA y MQA. Llama-3/Mistral/Gemma usan GQA → KV cache 4-8x más chica → throughput de inference mucho mayor.
- vLLM, TGI y SGLang combinan los 3 + PagedAttention para servir LLMs eficientemente.